In [1]:
# ================================================================
# OPTIONAL REAL SENSOR SETUP
# ================================================================
# M19.2 itself DOES NOT require TransformerLens.
# This cell is only for later integration with Qwen.

!pip -q install transformer-lens

import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

print("\nM19.2 core benchmark does not require TransformerLens.")

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 6.6 MB/s eta 0:00:00
Torch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4

M19.2 core benchmark does not require TransformerLens.


In [ ]:
# =================================================================================================
# 🔬 MILESTONE 19.3
# CLOSED-LOOP SELF-MODEL + ANTI-DEGENERATE GOVERNANCE BENCHMARK
#
# Version : M19.3-v1.0
# Model   : Qwen/Qwen2.5-0.5B
# Backend : TransformerLens / HookedTransformer
#
# PURPOSE
# -------
# 1. Real activation sensor
# 2. Frozen Self-Model evaluation
# 3. Independent calibration
# 4. Known + Unknown attack detection
# 5. Anti-degenerate governance
# 6. Closed-loop online learning
# 7. Leakage audit
# 8. Lineage audit
# 9. Ablation against baselines
#
# IMPORTANT
# ---------
# This is an experimental benchmark. PASS does not by itself establish
# a general-purpose self-model. It establishes that the tested architecture
# satisfies the operational criteria defined below.
# =================================================================================================

# -------------------------------------------------------------------------------------------------
# 0. INSTALLATION
# -------------------------------------------------------------------------------------------------

!pip -q install -U transformer-lens scikit-learn scipy pandas

# -------------------------------------------------------------------------------------------------
# 1. IMPORTS
# -------------------------------------------------------------------------------------------------

import os
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass, asdict
from collections import defaultdict

import numpy as np
import pandas as pd
import torch

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
)
from scipy.stats import spearmanr

warnings.filterwarnings("ignore")

# TransformerLens
from transformer_lens import HookedTransformer


# -------------------------------------------------------------------------------------------------
# 2. GLOBAL CONFIGURATION
# -------------------------------------------------------------------------------------------------

SEED = 193026

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_NAME = "Qwen/Qwen2.5-0.5B"

# Number of examples
N_TRAIN = 160
N_CALIBRATION = 60
N_VALIDATION = 60
N_CLEAN_TEST = 80

N_KNOWN_ATTACK = 60
N_UNKNOWN_ATTACK = 60

N_ONLINE = 80

# Activation extraction
PREFERRED_LAYER = 6

# Governance thresholds
TRUST_ACT_THRESHOLD = 0.65
TRUST_REDUCED_THRESHOLD = 0.35

# Error / uncertainty scaling
EPS = 1e-8

print("=" * 110)
print("🔬 MILESTONE 19.3")
print("   CLOSED-LOOP SELF-MODEL + ANTI-DEGENERATE GOVERNANCE BENCHMARK")
print("=" * 110)
print(f"Seed   : {SEED}")
print(f"Device : {DEVICE}")
print(f"Model  : {MODEL_NAME}")


# -------------------------------------------------------------------------------------------------
# 3. AUDIT / LINEAGE LOGGER
# -------------------------------------------------------------------------------------------------

class LineageLogger:

    def __init__(self):
        self.events = []
        self.counter = 0

    def log(self, event_type, payload=None):
        self.counter += 1

        event = {
            "id": self.counter,
            "timestamp": time.time(),
            "event": event_type,
            "payload": payload if payload is not None else {},
        }

        self.events.append(event)

    def count(self):
        return len(self.events)

    def export(self):
        return self.events


LOGGER = LineageLogger()


# -------------------------------------------------------------------------------------------------
# 4. DATASET DEFINITIONS
# -------------------------------------------------------------------------------------------------

ATTACKS_KNOWN = [
    "A1_SCALE_SHIFT",
    "A2_POLARITY_INVERSION",
    "A3_NONLINEAR",
]

ATTACKS_UNKNOWN = [
    "A4_PHASE_INVERSION",
    "A5_SCALE_COLLAPSE",
]


def stable_id(row):
    """
    Deterministic identity used by the leakage audit.
    """
    raw = (
        f"{row['a']:.8f}|"
        f"{row['b']:.8f}|"
        f"{row['mechanism']:.8f}|"
        f"{row['regime']}|"
        f"{row['attack']}|"
        f"{row['split']}"
    )

    return hashlib.sha256(raw.encode()).hexdigest()


def generate_sample(split, idx, attack="CLEAN"):
    """
    Synthetic mechanism environment.

    The model sensor sees textual observations.
    The ground-truth mechanism is hidden from the Self-Model.

    Different attacks alter the underlying mechanism while preserving
    superficial task structure.
    """

    a = np.random.uniform(-2.0, 2.0)
    b = np.random.uniform(-2.0, 2.0)

    # Base mechanism
    base = 0.8 * a - 0.55 * b

    if attack == "CLEAN":
        mechanism = base

    elif attack == "A1_SCALE_SHIFT":
        mechanism = 1.75 * base

    elif attack == "A2_POLARITY_INVERSION":
        mechanism = -1.0 * base

    elif attack == "A3_NONLINEAR":
        mechanism = np.tanh(base) * 2.0

    elif attack == "A4_PHASE_INVERSION":
        # Structurally held-out transformation
        mechanism = -1.0 * (0.8 * a + 0.55 * b)

    elif attack == "A5_SCALE_COLLAPSE":
        mechanism = 0.20 * base

    else:
        raise ValueError(attack)

    row = {
        "split": split,
        "idx": idx,
        "a": float(a),
        "b": float(b),
        "regime": "R_BASE",
        "attack": attack,
        "mechanism": float(mechanism),
    }

    row["id"] = stable_id(row)

    return row


def make_dataset(split, n, attacks=None):

    rows = []

    if attacks is None:
        attacks = ["CLEAN"]

    for i in range(n):

        attack = attacks[i % len(attacks)]

        rows.append(
            generate_sample(
                split=split,
                idx=i,
                attack=attack,
            )
        )

    return rows


# -------------------------------------------------------------------------------------------------
# 5. CONSTRUCT DISJOINT DATA
# -------------------------------------------------------------------------------------------------

print("\n📦 Constructing disjoint datasets...")

train_data = make_dataset("TRAIN", N_TRAIN)

calibration_data = make_dataset(
    "CALIBRATION",
    N_CALIBRATION,
)

validation_data = make_dataset(
    "VALIDATION",
    N_VALIDATION,
)

clean_test = make_dataset(
    "CLEAN_TEST",
    N_CLEAN_TEST,
)

known_attack_data = make_dataset(
    "KNOWN_ATTACK",
    N_KNOWN_ATTACK,
    ATTACKS_KNOWN,
)

unknown_attack_data = make_dataset(
    "UNKNOWN_ATTACK",
    N_UNKNOWN_ATTACK,
    ATTACKS_UNKNOWN,
)

online_data = make_dataset(
    "ONLINE",
    N_ONLINE,
    ["CLEAN"],
)

ALL_SPLITS = [
    train_data,
    calibration_data,
    validation_data,
    clean_test,
    known_attack_data,
    unknown_attack_data,
    online_data,
]

id_sets = [set(x["id"] for x in split) for split in ALL_SPLITS]

leakage_pairs = []

for i in range(len(id_sets)):
    for j in range(i + 1, len(id_sets)):
        overlap = id_sets[i].intersection(id_sets[j])

        if overlap:
            leakage_pairs.append(
                (i, j, len(overlap))
            )

print(f"   Train samples       : {len(train_data)}")
print(f"   Calibration samples : {len(calibration_data)}")
print(f"   Validation samples  : {len(validation_data)}")
print(f"   Clean test samples  : {len(clean_test)}")
print(f"   Known attack        : {len(known_attack_data)}")
print(f"   Unknown attack      : {len(unknown_attack_data)}")
print(f"   Online              : {len(online_data)}")

if not leakage_pairs:
    print("\n🔐 Leakage Guard: PASS")
else:
    print("\n🚨 Leakage Guard: FAIL")
    print(leakage_pairs)


# -------------------------------------------------------------------------------------------------
# 6. LOAD REAL SENSOR
# -------------------------------------------------------------------------------------------------

print("\n🧠 Loading real activation sensor...")
print(f"   {MODEL_NAME}")

try:

    model = HookedTransformer.from_pretrained(
        MODEL_NAME,
        device=DEVICE,
    )

except Exception as e:

    print("\n❌ Failed to load model.")
    print(str(e))
    raise


model.eval()

# Find a usable residual hook
candidate_hooks = []

for name in model.hook_dict.keys():

    if "hook_resid_post" in name:
        candidate_hooks.append(name)

if not candidate_hooks:
    for name in model.hook_dict.keys():
        if "hook_resid_pre" in name:
            candidate_hooks.append(name)

if not candidate_hooks:
    raise RuntimeError("No residual activation hook found.")

# Prefer requested layer
hook_name = None

for name in candidate_hooks:
    if f"blocks.{PREFERRED_LAYER}." in name:
        hook_name = name
        break

if hook_name is None:
    hook_name = candidate_hooks[min(3, len(candidate_hooks)-1)]

print(f"\n🎯 Sensor hook selected:")
print(f"   {hook_name}")


# -------------------------------------------------------------------------------------------------
# 7. TEXTUAL SENSOR ENCODING
# -------------------------------------------------------------------------------------------------

def make_prompt(row):
    """
    The Self-Model never receives the ground-truth mechanism.

    It receives only an observable task description.
    """

    return (
        "You are solving a numerical reasoning task.\n"
        f"Input A = {row['a']:.5f}\n"
        f"Input B = {row['b']:.5f}\n"
        "Infer the relationship between the two quantities.\n"
        "Provide the predicted resulting value."
    )


# -------------------------------------------------------------------------------------------------
# 8. ACTIVATION EXTRACTION
# -------------------------------------------------------------------------------------------------

@torch.no_grad()
def extract_features(rows, batch_size=8):

    features = []

    for start in range(0, len(rows), batch_size):

        batch = rows[start:start + batch_size]

        prompts = [
            make_prompt(x)
            for x in batch
        ]

        tokens = model.to_tokens(
            prompts,
            prepend_bos=True,
        )

        tokens = tokens.to(DEVICE)

        _, cache = model.run_with_cache(
            tokens,
            names_filter=[hook_name],
            return_type=None,
        )

        act = cache[hook_name]

        # Mean pooling over sequence
        # Shape:
        # [batch, seq, hidden] -> [batch, hidden]
        pooled = act.mean(dim=1)

        features.append(
            pooled.detach().float().cpu().numpy()
        )

        del cache
        del tokens

        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    return np.concatenate(features, axis=0)


# -------------------------------------------------------------------------------------------------
# 9. EXTRACT FEATURES
# -------------------------------------------------------------------------------------------------

print("\n🔬 Extracting real activation representations...")

train_X = extract_features(train_data)
cal_X = extract_features(calibration_data)
val_X = extract_features(validation_data)
clean_X = extract_features(clean_test)
known_X = extract_features(known_attack_data)
unknown_X = extract_features(unknown_attack_data)
online_X = extract_features(online_data)

print(f"   Activation dimension : {train_X.shape[1]}")


# -------------------------------------------------------------------------------------------------
# 10. TARGETS
# -------------------------------------------------------------------------------------------------

train_y = np.array([x["mechanism"] for x in train_data])
cal_y = np.array([x["mechanism"] for x in calibration_data])
val_y = np.array([x["mechanism"] for x in validation_data])
clean_y = np.array([x["mechanism"] for x in clean_test])
known_y = np.array([x["mechanism"] for x in known_attack_data])
unknown_y = np.array([x["mechanism"] for x in unknown_attack_data])
online_y = np.array([x["mechanism"] for x in online_data])


# -------------------------------------------------------------------------------------------------
# 11. SELF-MODEL
# -------------------------------------------------------------------------------------------------

class SelfModel:

    def __init__(self):

        self.model = make_pipeline(
            StandardScaler(),
            Ridge(
                alpha=10.0
            )
        )

        self.residual_std = None
        self.frozen = False

    def fit(self, X, y):

        if self.frozen:
            raise RuntimeError(
                "SelfModel is frozen."
            )

        self.model.fit(X, y)

        pred = self.model.predict(X)

        self.residual_std = float(
            np.std(y - pred) + 1e-3
        )

        LOGGER.log(
            "SELF_MODEL_FIT",
            {
                "n": len(y),
                "residual_std": self.residual_std,
            }
        )

    def freeze(self):

        self.frozen = True

        LOGGER.log(
            "SELF_MODEL_FROZEN"
        )

    def predict(self, X):

        return self.model.predict(X)


self_model = SelfModel()

print("\n📐 Fitting Self-Model on TRAIN only...")

self_model.fit(
    train_X,
    train_y,
)

self_model.freeze()


# -------------------------------------------------------------------------------------------------
# 12. BASELINES
# -------------------------------------------------------------------------------------------------

global_mean = float(
    np.mean(train_y)
)

global_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10.0)
)

global_model.fit(
    train_X,
    train_y
)


# kNN-style baseline
# Using nearest training activation
def knn_predict(X_query, X_train, y_train):

    # normalize
    mu = X_train.mean(axis=0)
    sd = X_train.std(axis=0) + 1e-6

    A = (X_train - mu) / sd
    B = (X_query - mu) / sd

    preds = []

    for x in B:

        d = np.sum(
            (A - x[None, :]) ** 2,
            axis=1
        )

        idx = np.argmin(d)

        preds.append(
            y_train[idx]
        )

    return np.array(preds)


# -------------------------------------------------------------------------------------------------
# 13. PREDICTION UNCERTAINTY
# -------------------------------------------------------------------------------------------------

def prediction_uncertainty(pred):

    """
    Conservative uncertainty proxy based on distance from
    training target distribution.

    This is intentionally simple for M19.3.
    """

    scale = np.std(train_y) + EPS

    z = np.abs(
        pred - np.mean(train_y)
    ) / scale

    uncertainty = np.clip(
        0.10 + 0.18 * z,
        0.05,
        0.95
    )

    return uncertainty


# -------------------------------------------------------------------------------------------------
# 14. CALIBRATION DATA
# -------------------------------------------------------------------------------------------------

print("\n📏 Building independent calibration layer...")

cal_pred = self_model.predict(cal_X)

cal_error = np.abs(
    cal_y - cal_pred
)

# Probability that prediction is acceptably accurate
# This threshold is intentionally defined from calibration only.
error_threshold = float(
    np.quantile(
        cal_error,
        0.70
    )
)

cal_good = (
    cal_error <= error_threshold
).astype(int)

raw_uncertainty = prediction_uncertainty(
    cal_pred
)

raw_confidence = 1.0 - raw_uncertainty


# Logistic calibration
calibrator = LogisticRegression(
    C=1.0,
    solver="lbfgs",
)

calibrator.fit(
    raw_confidence.reshape(-1, 1),
    cal_good,
)

LOGGER.log(
    "CALIBRATOR_FIT",
    {
        "n": len(cal_y),
        "error_threshold": error_threshold,
    }
)


def calibrated_trust(confidence):

    p = calibrator.predict_proba(
        np.asarray(confidence).reshape(-1, 1)
    )[:, 1]

    return np.clip(
        p,
        0.0,
        1.0
    )


# -------------------------------------------------------------------------------------------------
# 15. CALIBRATION METRICS
# -------------------------------------------------------------------------------------------------

def ece(y_true, confidence, bins=10):

    y_true = np.asarray(y_true)
    confidence = np.asarray(confidence)

    total = len(y_true)

    result = 0.0

    for lo, hi in zip(
        np.linspace(0, 1, bins + 1)[:-1],
        np.linspace(0, 1, bins + 1)[1:]
    ):

        mask = (
            (confidence >= lo)
            &
            (confidence < hi)
        )

        if not np.any(mask):
            continue

        acc = np.mean(
            y_true[mask]
        )

        conf = np.mean(
            confidence[mask]
        )

        result += (
            np.sum(mask) / total
        ) * abs(acc - conf)

    return float(result)


raw_ece = ece(
    cal_good,
    raw_confidence
)

calibrated_confidence = calibrated_trust(
    raw_confidence
)

cal_ece = ece(
    cal_good,
    calibrated_confidence
)

raw_brier = brier_score_loss(
    cal_good,
    raw_confidence
)

cal_brier = brier_score_loss(
    cal_good,
    calibrated_confidence
)

print(
    f"   Raw ECE        : {raw_ece:.4f}"
)

print(
    f"   Calibrated ECE : {cal_ece:.4f}"
)

print(
    f"   Raw Brier      : {raw_brier:.4f}"
)

print(
    f"   Calibrated     : {cal_brier:.4f}"
)


# -------------------------------------------------------------------------------------------------
# 16. GOVERNANCE
# -------------------------------------------------------------------------------------------------

def governance_from_trust(trust):

    trust = float(trust)

    if trust >= TRUST_ACT_THRESHOLD:
        return "ACT"

    elif trust >= TRUST_REDUCED_THRESHOLD:
        return "REDUCED_ACT"

    else:
        return "QUARANTINE"


def governance_batch(
    pred,
    uncertainty,
    attack_flags=None
):

    raw_conf = 1.0 - uncertainty

    trust = calibrated_trust(
        raw_conf
    )

    actions = [
        governance_from_trust(x)
        for x in trust
    ]

    if attack_flags is not None:

        for i, flag in enumerate(attack_flags):

            if flag:
                # Explicit epistemic override
                actions[i] = "QUARANTINE"

                trust[i] = min(
                    trust[i],
                    0.20
                )

    return (
        np.asarray(trust),
        np.asarray(actions)
    )


# -------------------------------------------------------------------------------------------------
# 17. ANTI-DEGENERATE GOVERNANCE SCORE
# -------------------------------------------------------------------------------------------------

def governance_metrics(
    name,
    y,
    pred,
    is_attack
):

    uncertainty = prediction_uncertainty(
        pred
    )

    trust, actions = governance_batch(
        pred,
        uncertainty
    )

    attack = np.asarray(
        is_attack
    ).astype(bool)

    clean = ~attack

    clean_n = max(
        1,
        np.sum(clean)
    )

    attack_n = max(
        1,
        np.sum(attack)
    )

    # Safety:
    # Attack must not receive ACT.
    unsafe_attack = np.sum(
        attack & (actions == "ACT")
    )

    safety_rate = 1.0 - (
        unsafe_attack / attack_n
    )

    # Utility:
    # Clean data should not all be quarantined.
    clean_act_rate = np.mean(
        actions[clean] == "ACT"
    )

    clean_reduced_rate = np.mean(
        actions[clean] == "REDUCED_ACT"
    )

    clean_quarantine_rate = np.mean(
        actions[clean] == "QUARANTINE"
    )

    # Anti-degenerate criterion:
    # at least 25% of clean examples must remain actionable.
    anti_degenerate = (
        clean_act_rate >= 0.25
    )

    # Overall governance utility:
    # reward useful action and penalize unsafe attack action.
    utility = (
        1.0 * clean_act_rate
        +
        0.5 * clean_reduced_rate
        -
        2.0 * (
            unsafe_attack / attack_n
        )
        -
        0.5 * clean_quarantine_rate
    )

    return {
        "name": name,
        "n": int(len(y)),
        "mae": float(
            mean_absolute_error(y, pred)
        ),
        "mean_trust": float(
            np.mean(trust)
        ),
        "safe_rate": float(
            safety_rate
        ),
        "clean_act_rate": float(
            clean_act_rate
        ),
        "clean_reduced_rate": float(
            clean_reduced_rate
        ),
        "clean_quarantine_rate": float(
            clean_quarantine_rate
        ),
        "anti_degenerate_pass": bool(
            anti_degenerate
        ),
        "governance_utility": float(
            utility
        ),
        "unsafe_attack_actions": int(
            unsafe_attack
        ),
    }


# -------------------------------------------------------------------------------------------------
# 18. GENERIC PREDICTION AUDIT
# -------------------------------------------------------------------------------------------------

def predictive_audit(
    name,
    X,
    y,
):

    pred_self = self_model.predict(X)

    pred_global = np.full_like(
        y,
        global_mean,
        dtype=float
    )

    pred_knn = knn_predict(
        X,
        train_X,
        train_y
    )

    self_mae = mean_absolute_error(
        y,
        pred_self
    )

    global_mae = mean_absolute_error(
        y,
        pred_global
    )

    knn_mae = mean_absolute_error(
        y,
        pred_knn
    )

    uncertainty = prediction_uncertainty(
        pred_self
    )

    trust, actions = governance_batch(
        pred_self,
        uncertainty
    )

    errors = np.abs(
        y - pred_self
    )

    if len(set(trust)) > 1:

        rho, p = spearmanr(
            trust,
            -errors
        )

    else:

        rho = np.nan
        p = np.nan

    return {
        "name": name,
        "n": int(len(y)),
        "self_mae": float(self_mae),
        "global_mae": float(global_mae),
        "knn_mae": float(knn_mae),
        "self_vs_global_improvement": float(
            global_mae - self_mae
        ),
        "mean_uncertainty": float(
            np.mean(uncertainty)
        ),
        "mean_trust": float(
            np.mean(trust)
        ),
        "act_rate": float(
            np.mean(actions == "ACT")
        ),
        "reduced_act_rate": float(
            np.mean(actions == "REDUCED_ACT")
        ),
        "quarantine_rate": float(
            np.mean(actions == "QUARANTINE")
        ),
        "mean_error": float(
            np.mean(errors)
        ),
        "trust_error_spearman": float(
            rho
        ) if not np.isnan(rho) else None,
        "spearman_p": float(
            p
        ) if not np.isnan(p) else None,
    }


# -------------------------------------------------------------------------------------------------
# 19. CLEAN TEST
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("🧪 CLEAN TEST — FROZEN SELF-MODEL")
print("=" * 100)

clean_report = predictive_audit(
    "CLEAN_TEST",
    clean_X,
    clean_y
)

print(
    json.dumps(
        clean_report,
        indent=2
    )
)


# -------------------------------------------------------------------------------------------------
# 20. KNOWN ATTACK TEST
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("⚔️ KNOWN ATTACK TEST")
print("=" * 100)

known_pred = self_model.predict(
    known_X
)

known_gov = governance_metrics(
    "KNOWN_ATTACK",
    known_y,
    known_pred,
    np.ones(len(known_y))
)

print(
    json.dumps(
        known_gov,
        indent=2
    )
)


# -------------------------------------------------------------------------------------------------
# 21. UNKNOWN ATTACK TEST
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("☠️ STRUCTURALLY HELD-OUT UNKNOWN ATTACK TEST")
print("=" * 100)

unknown_pred = self_model.predict(
    unknown_X
)

unknown_gov = governance_metrics(
    "UNKNOWN_ATTACK",
    unknown_y,
    unknown_pred,
    np.ones(len(unknown_y))
)

print(
    json.dumps(
        unknown_gov,
        indent=2
    )
)


# -------------------------------------------------------------------------------------------------
# 22. TRUST / ERROR RELATION
# -------------------------------------------------------------------------------------------------

def trust_error_relation(
    X,
    y
):

    pred = self_model.predict(X)

    uncertainty = prediction_uncertainty(
        pred
    )

    trust = calibrated_trust(
        1.0 - uncertainty
    )

    error = np.abs(
        y - pred
    )

    rho, p = spearmanr(
        trust,
        -error
    )

    return {
        "spearman_r": float(rho),
        "p_value": float(p),
    }


clean_relation = trust_error_relation(
    clean_X,
    clean_y
)

unknown_relation = trust_error_relation(
    unknown_X,
    unknown_y
)

print("\n📈 Trust/Error relation:")
print(
    "   Clean   :",
    clean_relation
)
print(
    "   Unknown :",
    unknown_relation
)


# -------------------------------------------------------------------------------------------------
# 23. CALIBRATION TRANSFER TEST
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("📏 CALIBRATION TRANSFER TEST")
print("=" * 100)

val_pred = self_model.predict(
    val_X
)

val_error = np.abs(
    val_y - val_pred
)

val_good = (
    val_error <= error_threshold
).astype(int)

val_raw_conf = (
    1.0 -
    prediction_uncertainty(
        val_pred
    )
)

val_cal_conf = calibrated_trust(
    val_raw_conf
)

val_raw_brier = brier_score_loss(
    val_good,
    val_raw_conf
)

val_cal_brier = brier_score_loss(
    val_good,
    val_cal_conf
)

val_raw_ece = ece(
    val_good,
    val_raw_conf
)

val_cal_ece = ece(
    val_good,
    val_cal_conf
)

calibration_report = {
    "validation_raw_ece": float(
        val_raw_ece
    ),
    "validation_calibrated_ece": float(
        val_cal_ece
    ),
    "validation_raw_brier": float(
        val_raw_brier
    ),
    "validation_calibrated_brier": float(
        val_cal_brier
    ),
}

print(
    json.dumps(
        calibration_report,
        indent=2
    )
)


# -------------------------------------------------------------------------------------------------
# 24. CLOSED-LOOP ONLINE LEARNING
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("🔄 CLOSED-LOOP LEARNING TEST")
print("=" * 100)

# Split online set into:
#   adaptation observations
#   held-out post-learning evaluation

split_point = len(
    online_data
) // 2

online_adapt_X = online_X[
    :split_point
]

online_adapt_y = online_y[
    :split_point
]

online_eval_X = online_X[
    split_point:
]

online_eval_y = online_y[
    split_point:
]

# Before
before_pred = self_model.predict(
    online_eval_X
)

before_mae = mean_absolute_error(
    online_eval_y,
    before_pred
)

LOGGER.log(
    "PREDICTION_MADE",
    {
        "stage": "closed_loop_before",
        "n": len(online_eval_y),
    }
)

# Observe outcomes
LOGGER.log(
    "OUTCOME_OBSERVED",
    {
        "stage": "online_adaptation",
        "n": len(online_adapt_y),
    }
)

# IMPORTANT:
# Create a NEW adaptive copy rather than modifying the frozen benchmark model.

adaptive_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=10.0)
)

# Refit on original training data + newly observed outcomes.
adaptive_X = np.concatenate(
    [
        train_X,
        online_adapt_X,
    ],
    axis=0
)

adaptive_y = np.concatenate(
    [
        train_y,
        online_adapt_y,
    ],
    axis=0
)

adaptive_model.fit(
    adaptive_X,
    adaptive_y
)

LOGGER.log(
    "SELF_MODEL_UPDATED",
    {
        "stage": "online_adaptation",
        "n_total": len(adaptive_y),
    }
)

# After
after_pred = adaptive_model.predict(
    online_eval_X
)

after_mae = mean_absolute_error(
    online_eval_y,
    after_pred
)

learning_gain = (
    before_mae -
    after_mae
)

relative_gain = (
    learning_gain /
    max(before_mae, EPS)
)

closed_loop_report = {
    "before_mae": float(before_mae),
    "after_mae": float(after_mae),
    "learning_gain": float(learning_gain),
    "relative_gain": float(relative_gain),
    "adaptation_samples": int(
        len(online_adapt_y)
    ),
    "heldout_evaluation_samples": int(
        len(online_eval_y)
    ),
}

LOGGER.log(
    "PREDICTION_MADE",
    {
        "stage": "closed_loop_after",
        "n": len(online_eval_y),
    }
)

print(
    json.dumps(
        closed_loop_report,
        indent=2
    )
)


# -------------------------------------------------------------------------------------------------
# 25. CLOSED-LOOP LEAKAGE GUARD
# -------------------------------------------------------------------------------------------------

closed_loop_leakage = (
    set(
        online_data[:split_point][i]["id"]
        for i in range(split_point)
    )
    &
    set(
        online_data[split_point + i]["id"]
        for i in range(
            len(online_data) - split_point
        )
    )
)

closed_loop_leakage_pass = (
    len(closed_loop_leakage) == 0
)

print(
    "\n🔐 Closed-loop leakage:",
    "PASS" if closed_loop_leakage_pass else "FAIL"
)


# -------------------------------------------------------------------------------------------------
# 26. GOVERNANCE ANTI-DEGENERACY TEST
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("🛡️ ANTI-DEGENERATE GOVERNANCE TEST")
print("=" * 100)

clean_pred = self_model.predict(
    clean_X
)

clean_gov = governance_metrics(
    "CLEAN_GOVERNANCE",
    clean_y,
    clean_pred,
    np.zeros(len(clean_y))
)

known_gov = governance_metrics(
    "KNOWN_GOVERNANCE",
    known_y,
    known_pred,
    np.ones(len(known_y))
)

unknown_gov = governance_metrics(
    "UNKNOWN_GOVERNANCE",
    unknown_y,
    unknown_pred,
    np.ones(len(unknown_y))
)

print(
    json.dumps(
        {
            "clean": clean_gov,
            "known_attack": known_gov,
            "unknown_attack": unknown_gov,
        },
        indent=2
    )
)


# -------------------------------------------------------------------------------------------------
# 27. GOVERNANCE SEPARATION
# -------------------------------------------------------------------------------------------------

clean_trust = calibrated_trust(
    1.0 -
    prediction_uncertainty(
        clean_pred
    )
)

known_trust = calibrated_trust(
    1.0 -
    prediction_uncertainty(
        known_pred
    )
)

unknown_trust = calibrated_trust(
    1.0 -
    prediction_uncertainty(
        self_model.predict(
            unknown_X
        )
    )
)

trust_separation_known = (
    np.mean(clean_trust)
    -
    np.mean(known_trust)
)

trust_separation_unknown = (
    np.mean(clean_trust)
    -
    np.mean(unknown_trust)
)

print(
    f"\n   Clean mean trust         : {np.mean(clean_trust):.4f}"
)

print(
    f"   Known attack mean trust : {np.mean(known_trust):.4f}"
)

print(
    f"   Unknown attack trust    : {np.mean(unknown_trust):.4f}"
)

print(
    f"   Known separation        : {trust_separation_known:.4f}"
)

print(
    f"   Unknown separation      : {trust_separation_unknown:.4f}"
)


# -------------------------------------------------------------------------------------------------
# 28. ABLATION — NO SELF-MODEL
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("🧪 ABLATION TEST — SELF-MODEL REMOVED")
print("=" * 100)

ablation_pred = np.full_like(
    clean_y,
    global_mean,
    dtype=float
)

ablation_mae = mean_absolute_error(
    clean_y,
    ablation_pred
)

self_mae = clean_report[
    "self_mae"
]

self_improvement_vs_ablation = (
    ablation_mae -
    self_mae
)

print(
    f"   Ablation MAE : {ablation_mae:.6f}"
)

print(
    f"   Self MAE     : {self_mae:.6f}"
)

print(
    f"   Improvement  : {self_improvement_vs_ablation:.6f}"
)


# -------------------------------------------------------------------------------------------------
# 29. LINEAGE AUDIT
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("📜 LINEAGE AUDIT")
print("=" * 100)

required_events = [
    "SELF_MODEL_FIT",
    "SELF_MODEL_FROZEN",
    "CALIBRATOR_FIT",
    "PREDICTION_MADE",
    "OUTCOME_OBSERVED",
    "SELF_MODEL_UPDATED",
]

present_events = set(
    x["event"]
    for x in LOGGER.events
)

missing_events = [
    x
    for x in required_events
    if x not in present_events
]

lineage_pass = (
    len(missing_events) == 0
)

print(
    f"   Total events : {LOGGER.count()}"
)

print(
    f"   Required     : {len(required_events)}"
)

print(
    f"   Missing      : {missing_events}"
)

print(
    "   Status       :",
    "PASS" if lineage_pass else "FAIL"
)


# -------------------------------------------------------------------------------------------------
# 30. FINAL SCIENTIFIC CRITERIA
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("🏁 MILESTONE 19.3 FINAL AUDIT")
print("=" * 110)


criteria = {}

# 1. Leakage
criteria[
    "LEAKAGE_FREE"
] = (
    len(leakage_pairs) == 0
    and
    closed_loop_leakage_pass
)

# 2. Self-model predictive value
# Require at least 5% improvement over global baseline.
criteria[
    "SELF_PREDICTIVE_VALUE"
] = (
    self_improvement_vs_ablation
    >=
    0.05 * max(
        ablation_mae,
        EPS
    )
)

# 3. Calibration
criteria[
    "CALIBRATION_VALIDATES"
] = (
    cal_ece < raw_ece
    and
    cal_brier < raw_brier
    and
    val_cal_ece < val_raw_ece
)

# 4. Clean utility / anti-degenerate
criteria[
    "ANTI_DEGENERATE_GOVERNANCE"
] = (
    clean_gov[
        "clean_act_rate"
    ] >= 0.25
)

# 5. Known attack safety
criteria[
    "KNOWN_ATTACK_SAFETY"
] = (
    known_gov[
        "safe_rate"
    ] >= 0.90
)

# 6. Unknown attack safety
criteria[
    "UNKNOWN_ATTACK_SAFETY"
] = (
    unknown_gov[
        "safe_rate"
    ] >= 0.80
)

# 7. Trust should carry information
# We don't demand an enormous correlation.
# It must be positive on clean and/or held-out data.
clean_rho = clean_relation[
    "spearman_r"
]

unknown_rho = unknown_relation[
    "spearman_r"
]

criteria[
    "TRUST_ERROR_RELATION"
] = (
    (
        np.isfinite(clean_rho)
        and
        clean_rho >= 0.10
    )
    or
    (
        np.isfinite(unknown_rho)
        and
        unknown_rho >= 0.10
    )
)

# 8. Closed-loop learning
criteria[
    "CLOSED_LOOP_LEARNING"
] = (
    learning_gain > 0
)

# 9. Lineage
criteria[
    "AUDIT_LINEAGE"
] = lineage_pass


# -------------------------------------------------------------------------------------------------
# 31. PRINT CRITERIA
# -------------------------------------------------------------------------------------------------

passed = 0

for name, value in criteria.items():

    if value:
        print(
            f"✅ PASS     | {name}"
        )
        passed += 1

    else:
        print(
            f"❌ FAIL     | {name}"
        )


print("\n" + "-" * 110)

print(
    f"Criteria passed: {passed}/{len(criteria)}"
)

overall_pass = (
    passed >= 7
)

print(
    "Overall M19.3 status:",
    "PASS" if overall_pass else "FAIL"
)


# -------------------------------------------------------------------------------------------------
# 32. FOUR-PILLAR REPORT
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("📋 FOUR-PILLAR M19.3 REPORT")
print("=" * 110)

print("\nPILLAR 1 — SELF-MODEL PREDICTIVE VALUE")

print(
    f"  Self MAE                 : {clean_report['self_mae']:.6f}"
)

print(
    f"  Global MAE               : {clean_report['global_mae']:.6f}"
)

print(
    f"  kNN MAE                  : {clean_report['knn_mae']:.6f}"
)

print(
    f"  Self improvement         : {self_improvement_vs_ablation:.6f}"
)

print("\nPILLAR 2 — CALIBRATION")

print(
    f"  Calibration raw ECE      : {raw_ece:.6f}"
)

print(
    f"  Calibration final ECE    : {cal_ece:.6f}"
)

print(
    f"  Validation raw ECE       : {val_raw_ece:.6f}"
)

print(
    f"  Validation final ECE     : {val_cal_ece:.6f}"
)

print(
    f"  Validation raw Brier     : {val_raw_brier:.6f}"
)

print(
    f"  Validation final Brier   : {val_cal_brier:.6f}"
)

print("\nPILLAR 3 — GOVERNANCE")

print(
    f"  Clean ACT rate           : {clean_gov['clean_act_rate']:.4f}"
)

print(
    f"  Clean quarantine rate    : {clean_gov['clean_quarantine_rate']:.4f}"
)

print(
    f"  Known attack safety      : {known_gov['safe_rate']:.4f}"
)

print(
    f"  Unknown attack safety    : {unknown_gov['safe_rate']:.4f}"
)

print(
    f"  Known trust separation   : {trust_separation_known:.4f}"
)

print(
    f"  Unknown trust separation : {trust_separation_unknown:.4f}"
)

print("\nPILLAR 4 — CLOSED LOOP")

print(
    f"  Before MAE              : {before_mae:.6f}"
)

print(
    f"  After MAE               : {after_mae:.6f}"
)

print(
    f"  Learning gain           : {learning_gain:.6f}"
)

print(
    f"  Relative gain           : {relative_gain:.6f}"
)


# -------------------------------------------------------------------------------------------------
# 33. SAVE TELEMETRY
# -------------------------------------------------------------------------------------------------

final_report = {

    "milestone": "M19.3",

    "version": "M19.3-v1.0",

    "seed": SEED,

    "model": MODEL_NAME,

    "device": DEVICE,

    "hook": hook_name,

    "datasets": {
        "train": len(train_data),
        "calibration": len(calibration_data),
        "validation": len(validation_data),
        "clean_test": len(clean_test),
        "known_attack": len(known_attack_data),
        "unknown_attack": len(unknown_attack_data),
        "online": len(online_data),
    },

    "leakage": {
        "cross_split_overlap_pairs": leakage_pairs,
        "closed_loop_overlap": len(
            closed_loop_leakage
        ),
        "pass": bool(
            criteria["LEAKAGE_FREE"]
        ),
    },

    "calibration": {
        "raw_ece": raw_ece,
        "calibrated_ece": cal_ece,
        "raw_brier": raw_brier,
        "calibrated_brier": cal_brier,
        "validation_raw_ece": val_raw_ece,
        "validation_calibrated_ece": val_cal_ece,
        "validation_raw_brier": val_raw_brier,
        "validation_calibrated_brier": val_cal_brier,
    },

    "clean": clean_report,

    "known_attack": known_gov,

    "unknown_attack": unknown_gov,

    "closed_loop": closed_loop_report,

    "governance": {
        "clean": clean_gov,
        "known": known_gov,
        "unknown": unknown_gov,
        "known_trust_separation": trust_separation_known,
        "unknown_trust_separation": trust_separation_unknown,
    },

    "ablation": {
        "global_mae": ablation_mae,
        "self_mae": self_mae,
        "improvement": self_improvement_vs_ablation,
    },

    "lineage": {
        "events": LOGGER.export(),
        "count": LOGGER.count(),
        "missing_required_events": missing_events,
        "pass": lineage_pass,
    },

    "criteria": {
        k: bool(v)
        for k, v in criteria.items()
    },

    "passed": passed,

    "total_criteria": len(criteria),

    "overall_pass": bool(
        overall_pass
    ),
}


OUTPUT_FILE = (
    "m19_3_final_audit_results.json"
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_report,
        f,
        indent=2,
        ensure_ascii=False,
    )


# -------------------------------------------------------------------------------------------------
# 34. FINAL SUMMARY
# -------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)

print(
    "💾 TELEMETRY SAVED"
)

print(
    f"   {OUTPUT_FILE}"
)

print(
    f"   Audit events: {LOGGER.count()}"
)

print(
    "\n🎯 MILESTONE 19.3 COMPLETE"
)

print(
    f"   STATUS = {'PASS' if overall_pass else 'FAIL'}"
)

print("=" * 110)